# ডাবিং ইঞ্জিন (কোলাব)

এই নোটবুক আপনার ওয়েবসাইটের জন্য কোলাবকে "ইঞ্জিন" বানায়।

**ব্যবহার:** উপরের মেনু থেকে **Runtime → Run all** চাপুন (বা `Ctrl+F9`)। কয়েক মিনিট পর নিচে একটা সবুজ **"সাইটে কানেক্ট করুন"** লিংক আসবে। সেটায় ক্লিক করুন।

**প্রথমবার একবার:** বাঁ পাশের 🔑 **Secrets** এ `ENGINE_KEY` নামে একটা secret বানিয়ে আপনার key বসান এবং "Notebook access" অন করুন। এরপর থেকে শুধু Run all।

⚠️ এই ট্যাব বা উইন্ডো খোলা রাখুন। বন্ধ করলে কিছুক্ষণ পর সেশন কেটে যায়।

In [ ]:
#@title ১. সেটিংস { display-mode: "form" }
#@markdown নিজের Vercel সাইটের ঠিকানা দিন (শেষে / দেবেন না):
SITE_URL = "https://insafautodubbing.vercel.app"  #@param {type:"string"}

In [ ]:
#@title ২. সর্বশেষ ডাবিং কোড নামানো ও ইনস্টল { display-mode: "form" }
import json, os, shutil, subprocess, sys, urllib.error, urllib.request

SITE = SITE_URL.strip().rstrip("/")
if "YOUR-SITE" in SITE or not SITE.startswith("https://"):
    raise SystemExit("❌ উপরের ঘরে নিজের সাইটের ঠিকানা (https://...) দিন, তারপর আবার Run all করুন।")

# key: প্রথমে Colab Secrets থেকে, না থাকলে জিজ্ঞেস করবে
ENGINE_KEY = None
try:
    from google.colab import userdata
    ENGINE_KEY = userdata.get("ENGINE_KEY")
except Exception:
    pass
if not ENGINE_KEY:
    from getpass import getpass
    ENGINE_KEY = getpass("শুধু key টা পেস্ট করুন, তারপর Enter: ").strip()


# পেস্টের সময় ভুল করে অন্য লেখা (যেমন প্রম্পটের শব্দ) চলে এলেও শুধু key টা বেছে নেয়
import re
_found = re.findall(r"[A-Za-z0-9_\-]{30,}", ENGINE_KEY or "")
if not _found:
    raise SystemExit("❌ key ঠিকমতো পাওয়া যায়নি। শুধু key টা পেস্ট করুন (ইংরেজি অক্ষর, অঙ্ক, - আর _ দিয়ে লম্বা একটা লেখা)।")
ENGINE_KEY = _found[-1]

req = urllib.request.Request(
    SITE + "/api/engine",
    headers={"X-Engine-Key": ENGINE_KEY, "User-Agent": "colab-dubbing-engine/1.0"},
)
try:
    with urllib.request.urlopen(req, timeout=30) as r:
        data = json.loads(r.read().decode("utf-8"))
except urllib.error.HTTPError as e:
    if e.code == 401:
        raise SystemExit("❌ key মেলেনি। ENGINE_KEY ঠিক আছে কিনা দেখুন।")
    try:
        detail = e.read().decode("utf-8", "replace")[:400]
    except Exception:
        detail = ""
    raise SystemExit(f"❌ সাইট থেকে কোড আনা যায়নি (HTTP {e.code}): {detail}")
except Exception as e:
    raise SystemExit(f"❌ সাইটে পৌঁছানো যায়নি: {e}")

ENGINE_DIR = "/content/engine"
shutil.rmtree(ENGINE_DIR, ignore_errors=True)
os.makedirs(ENGINE_DIR)
for name, body in data["files"].items():
    if "/" in name or name.startswith("."):
        continue
    with open(os.path.join(ENGINE_DIR, name), "w", encoding="utf-8") as f:
        f.write(body)
ENGINE_VERSION = data["version"]
print("✅ কোড নামানো হয়েছে, ভার্সন:", ENGINE_VERSION)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ENGINE_DIR}/requirements.txt"], check=True)
if not shutil.which("ffmpeg"):
    subprocess.run("apt-get install -y -qq ffmpeg", shell=True, check=True)
print("✅ প্যাকেজ ইনস্টল শেষ")

In [ ]:
#@title ৩. সার্ভার ও টানেল চালু, তারপর সাইটের লিংক { display-mode: "form" }
import os, re, secrets, stat, subprocess, sys, threading, time, urllib.request
from urllib.parse import quote

SESSION_KEY = secrets.token_urlsafe(24)      # প্রতিবার নতুন গোপন key
os.environ["DUB_KEY"] = SESSION_KEY
os.environ["SITE_ORIGIN"] = SITE              # শুধু আপনার সাইট এই সার্ভারকে ডাকতে পারবে
os.environ["ENGINE_VERSION"] = ENGINE_VERSION

# --- সার্ভার ---
sys.path.insert(0, ENGINE_DIR)
for m in ("server", "process"):
    sys.modules.pop(m, None)
import uvicorn, server

PORT = 8000
srv = uvicorn.Server(uvicorn.Config(server.app, host="127.0.0.1", port=PORT, log_level="warning"))
threading.Thread(target=srv.run, daemon=True).start()
for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
        break
    except Exception:
        time.sleep(0.5)
else:
    raise SystemExit("❌ সার্ভার চালু হয়নি")
print("✅ সার্ভার চালু")

# --- Cloudflare টানেল ---
CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, os.stat(CF).st_mode | stat.S_IEXEC)

tunnel = subprocess.Popen(
    [CF, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

TUNNEL_URL = None
deadline = time.time() + 90
while time.time() < deadline and TUNNEL_URL is None:
    line = tunnel.stdout.readline()
    if not line:
        if tunnel.poll() is not None:
            break
        continue
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        TUNNEL_URL = m.group(0)
if not TUNNEL_URL:
    raise SystemExit("❌ টানেল চালু হয়নি। কিছুক্ষণ পর আবার Run all করুন।")
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()  # পাইপ খালি রাখি

LINK = f"{SITE}/#server={quote(TUNNEL_URL, safe=':/')}&key={SESSION_KEY}"
from IPython.display import HTML, display
display(HTML(
    "<div style='font-family:sans-serif;padding:14px;border-radius:10px;background:#e2f2ef;color:#0b4f45'>"
    "<div style='font-size:16px;margin-bottom:8px'>✅ ইঞ্জিন তৈরি। এই লিংকে ক্লিক করুন:</div>"
    f"<a href='{LINK}' target='_blank' rel='noopener' style='display:inline-block;padding:10px 18px;"
    "border-radius:8px;background:#0d7a6b;color:#fff;text-decoration:none;font-size:16px'>সাইটে কানেক্ট করুন →</a>"
    "<div style='margin-top:10px;font-size:13px'>এই উইন্ডো খোলা রাখুন। সাইট থেকে Disconnect চাপলে এটা নিজে বন্ধ হবে।</div>"
    "</div>"))
print("\nলিংক ক্লিক না হলে এটা কপি করে সাইটের 'লিংক পেস্ট করুন' ঘরে দিন:\n" + LINK + "\n")

# --- সাইট থেকে Disconnect না আসা পর্যন্ত চলতে থাকবে ---
try:
    while not server.STOP.is_set():
        time.sleep(2)
        if tunnel.poll() is not None:
            print("⚠️ টানেল বন্ধ হয়ে গেছে। আবার Run all করুন।")
            break
except KeyboardInterrupt:
    pass

tunnel.terminate()
srv.should_exit = True
print("🛑 বন্ধ হচ্ছে…")
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print("সেশন নিজে বন্ধ করা যায়নি, Runtime → Disconnect and delete runtime চাপুন।", e)